# FIXED COUPON BOND EXAMPLE APPLE CORP

This is a  bond analysis based on example in https://data.bloomberglp.com/bat/sites/3/2017/07/SF-2017_Paul-Fjeldsted.pdf

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from financepy.utils import *
from financepy.products.bonds.bond import *

#############################################################
#  FINANCEPY Version 1.1.2 - Built on 25 Sep 2026 at 14:07  #
#  This software is distributed FREE AND WITHOUT WARRANTY   #
#  Report issues at https://github.com/domokane/FinancePy   #
#############################################################



# Define the Bond

In [3]:
issue_dt = Date(13, 5, 2010)
maturity_dt = Date(13, 5, 2022)
coupon = 0.027
freq_type = FrequencyTypes.SEMI_ANNUAL
dc_type = DayCountTypes.THIRTY_E_360
face = ONE_MILLION

In [4]:
bond = Bond(issue_dt, maturity_dt, coupon, freq_type, dc_type)

In [5]:
clean_price = 101.581564  # if face is 1 then this must be 0.99780842

You can get information about the bond using the print method.

In [6]:
print(bond)

OBJECT_TYPE: Bond
ISSUE DATE: 13-MAY-2010
MATURITY_DATE: 13-MAY-2022
COUPON (%): 2.7
FREQUENCY: FrequencyTypes.SEMI_ANNUAL
DAY_COUNT: DayCountTypes.THIRTY_E_360
EX-DIVIDEND DAYS: 0
CALENDAR TYPE: CalendarTypes.WEEKEND
BUS DAYS ADJUST: BusDayAdjustTypes.FOLLOWING
DATE GEN RULE: DateGenRuleTypes.BACKWARD
COUPON TYPE: CouponType.FIXED


## Bond Cash Flows

We first need to set the settlement date of the bond. 

In [7]:
settle_dt = Date(21, 7, 2017)

In [8]:
bond.print_payments(settle_dt)

Coupon Date 	 Payment Date 	         Status          Amount
21-JUL-2017 	             	     SETTLEMENT 
13-NOV-2017 	 13-NOV-2017 	      UNCHANGED 	            1.35 
13-MAY-2018 	 14-MAY-2018 	   HOLIDAY ROLL 	            1.35 
13-NOV-2018 	 13-NOV-2018 	      UNCHANGED 	            1.35 
13-MAY-2019 	 13-MAY-2019 	      UNCHANGED 	            1.35 
13-NOV-2019 	 13-NOV-2019 	      UNCHANGED 	            1.35 
13-MAY-2020 	 13-MAY-2020 	      UNCHANGED 	            1.35 
13-NOV-2020 	 13-NOV-2020 	      UNCHANGED 	            1.35 
13-MAY-2021 	 13-MAY-2021 	      UNCHANGED 	            1.35 
13-NOV-2021 	 15-NOV-2021 	   HOLIDAY ROLL 	            1.35 
13-MAY-2022 	 13-MAY-2022 	       MATURITY 	          101.35 



The convention is to use these dates for yield calculations even if some fall on weekends.

## Bond Yield Measures

Current yield is the coupon over the price

In [9]:
bond.current_yield(settle_dt, clean_price)*100.0

np.float64(2.6446847263501616)

Yield to maturity using UK convention

In [10]:
bond.yield_to_maturity(settle_dt, clean_price, YTMCalcType.UK_DMO)*100

2.3499999794740978

Yield to maturity using US Street convention

In [11]:
bond.yield_to_maturity(settle_dt, clean_price, YTMCalcType.US_STREET)*100

2.3499999794740978

Yield to maturity using US Treasury convention

In [12]:
bond.yield_to_maturity(settle_dt, clean_price, YTMCalcType.US_TREASURY)*100

2.3496418129741814

## Accrued Interest

For consistency let's fix the yield calculation convention to be US Street

In [13]:
yieldConvention = YTMCalcType.US_STREET

In [14]:
ytm = bond.yield_to_maturity(settle_dt, clean_price, yieldConvention)

Dirty price is the clean price plus accrued interest

In [15]:
print("Dirty Price = %8.6f" %  bond.dirty_price_from_ytm(settle_dt, ytm, yieldConvention))

Dirty Price = 102.091564


In [16]:
print("Clean Price = %8.6f" % bond.clean_price_from_ytm(settle_dt, ytm, yieldConvention))

Clean Price = 101.581564


Accrued interest is accrued from previous coupon date to settlement date

In [17]:
print("Previous coupon date is ", bond._pcd)

Previous coupon date is  13-MAY-2017


In [18]:
print("Settlement date is ", settle_dt)

Settlement date is  21-JUL-2017


The amount of accrued interest is 

In [19]:
print("Accrued = %12.2f"% bond.accrued_int)

Accrued =         0.51


This is based on the following number of days of accrual (for some conventions this is not the actual number of days between last coupon and settlement)

In [20]:
print("Accrued Days = ", bond.accrued_days)

Accrued Days =  68


In [21]:
print("Principal = %12.2f" % bond.principal(settle_dt, ytm, face, yieldConvention))

Principal =   1015815.64


## Bond Risk Measures

The bond dollar duration is the actual derivative with respect to the yield. It is the bond price change for a 1bp drop in the yield-to-maturity divided by 1bp.

In [22]:
duration = bond.dollar_duration(settle_dt, ytm, yieldConvention)
print("Dollar Duration = ", duration)

Dollar Duration =  456.45180866003443


Modified Duration divides the dollar duration by the full price of the bond

In [23]:
modified_duration = bond.modified_duration(settle_dt, ytm, yieldConvention)
print("Modified Duration = ", modified_duration)

Modified Duration =  4.471004172897813


Macaulay Duration multiplies the dollar duration by (1+y/f) and divides by the full price

In [24]:
macaulay_duration = bond.macaulay_duration(settle_dt, ytm, yieldConvention)
print("Macaulay Duration = ", macaulay_duration)

Macaulay Duration =  4.523538471470506


Convexity is the second derivative of the bond price with respect to the yield-to-maturity

In [25]:
conv = bond.convexity_from_ytm(settle_dt, ytm, yieldConvention)
print("Convexity = ", conv)

Convexity =  23.024091262005694


Copyright (c) 2020 Dominic O'Kane